In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing
import os
import copy
import glob
import random
from random import random as rd
import gc




In [2]:
def _resolve_path(rel_path: str) -> str:
    candidates = [
        rel_path,
        rel_path.replace("../input/", "/kaggle/input/"),
        rel_path.replace("../input/", "/kaggle/data/"),
        rel_path.replace("../input/", "/kaggle/input/ventilator-pressure-prediction/"),
        rel_path.replace("../input/", "/kaggle/data/ventilator-pressure-prediction/"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return rel_path  # let pandas raise a clear error if nothing matches


train_path = _resolve_path("../input/ventilator-pressure-prediction/train.csv")
df_train = pd.read_csv(train_path)

unique_pressures = df_train["pressure"].unique()
sorted_pressures = np.sort(unique_pressures)
total_pressures_len = len(sorted_pressures)


def find_nearest(prediction):
    insert_idx = np.searchsorted(sorted_pressures, prediction)
    if insert_idx == total_pressures_len:
        return sorted_pressures[-1]
    elif insert_idx == 0:
        return sorted_pressures[0]
    lower_val = sorted_pressures[insert_idx - 1]
    upper_val = sorted_pressures[insert_idx]
    return (
        lower_val
        if abs(lower_val - prediction) < abs(upper_val - prediction)
        else upper_val
    )


def set_seed(seed=2021):
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    return random_state


def wc(input_list):
    l = []
    for i in range(len(input_list)):
        public_lb_score = int(input_list[i].split("/")[-1].split(".")[1].split(" ")[0])
        l.append(public_lb_score)
        input_list[i] = (pd.read_csv(input_list[i]).pressure).ravel()
    output = 0
    l_sum = sum(l)
    if len(input_list) == 1:
        output = input_list[0]
    else:
        weight1 = (l[1] / l_sum) + 0.05
        weight2 = 1 - weight1
        output += input_list[0] * weight1 + input_list[1] * weight2
    return output


def g(dp):
    l = []
    for i in glob.iglob(f"{dp}/*"):
        l.append(i)
    file_count = len(l)
    loop_time = 154
    splits = file_count // 2
    l.sort()
    flist = []
    for i in range(splits):
        if i == splits - 1:
            flist.append(l[i * round(len(l) / splits) :])
        else:
            flist.append(
                l[i * round(len(l) / splits) : (i + 1) * round(len(l) / splits)]
            )
    for i in range(len(flist)):
        flist[i] = wc(flist[i])
    pred_list = []
    for i in range(loop_time):
        weight = []
        set_seed(i)
        for i in range(len(flist)):
            weight.append(rd())
        weight_sum = sum(weight)
        for i in range(len(weight)):
            weight[i] /= weight_sum
        weight.sort(reverse=True)
        temp = 0
        for i in range(len(flist)):
            temp += flist[i] * weight[i]
        pred_list.append(temp)
        del temp
        gc.collect()
    output = pd.read_csv(
        _resolve_path("../input/ventilator-pressure-prediction/sample_submission.csv")
    )
    output.pressure = np.median(np.vstack(pred_list), axis=0)
    output["pressure"] = output["pressure"].apply(find_nearest)
    output.to_csv(f"rwb {loop_time} loops.csv", index=False)


def blend(a, b):
    a = pd.read_csv(a)
    b = pd.read_csv(b)
    a.pressure = a.pressure * 0.62 + b.pressure * 0.38
    a["pressure"] = a["pressure"].apply(find_nearest)
    a.to_csv("blend.csv", index=False)
    return a




In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge

test_path = _resolve_path("../input/ventilator-pressure-prediction/test.csv")
sample_path = _resolve_path(
    "../input/ventilator-pressure-prediction/sample_submission.csv"
)

df_test = pd.read_csv(test_path)
sub = pd.read_csv(sample_path)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.sort_values(["breath_id", "time_step"], inplace=True)

    df["u_in_lag1"] = df.groupby("breath_id")["u_in"].shift(1).fillna(0.0)
    df["u_in_lag2"] = df.groupby("breath_id")["u_in"].shift(2).fillna(0.0)

    df["u_in_diff1"] = df["u_in"] - df["u_in_lag1"]
    df["u_in_diff2"] = df["u_in_lag1"] - df["u_in_lag2"]

    df["u_in_cum"] = df.groupby("breath_id")["u_in"].cumsum()
    df["u_in_x_time"] = df["u_in"] * df["time_step"]

    df["dt"] = df.groupby("breath_id")["time_step"].diff().fillna(0.0)

    df["u_out_lag1"] = df.groupby("breath_id")["u_out"].shift(1).fillna(0).astype(int)
    df["u_out_cum"] = df.groupby("breath_id")["u_out"].cumsum().astype(float)

    df["u_in_area"] = (df["u_in"] * df["dt"]).groupby(df["breath_id"]).cumsum()

    df["u_in_over_R"] = df["u_in"] / df["R"].astype(float)
    df["u_in_over_C"] = df["u_in"] / df["C"].astype(float)
    df["time_over_R"] = df["time_step"] / df["R"].astype(float)
    df["time_over_C"] = df["time_step"] / df["C"].astype(float)

    df["is_insp"] = (df["u_out"] == 0).astype(int)

    tmin = df.groupby("breath_id")["time_step"].transform("min")
    tmax = df.groupby("breath_id")["time_step"].transform("max")
    denom = (tmax - tmin).replace(0, 1.0)
    df["t_norm"] = (df["time_step"] - tmin) / denom

    df["u_in_insp"] = df["u_in"] * (df["u_out"] == 0).astype(float)
    df["u_in_cum_insp"] = df.groupby("breath_id")["u_in_insp"].cumsum()

    df["t_idx"] = df.groupby("breath_id").cumcount().astype(np.int16)
    df["u_in_cum_sq"] = (df["u_in_cum"] ** 2).astype(float)

    u_out_change = (df["u_out"] != df["u_out_lag1"]).astype(np.int8)
    seg_id = u_out_change.groupby(df["breath_id"]).cumsum().astype(np.int16)

    first_time_in_seg = df.groupby(["breath_id", seg_id])["time_step"].transform(
        "first"
    )
    df["time_since_u_out_change"] = df["time_step"] - first_time_in_seg

    g = df.groupby("breath_id")

    df["insp_step"] = df["is_insp"].astype(np.int16)
    df["insp_count"] = g["insp_step"].cumsum().astype(np.int16)

    df["u_in_roll3_mean"] = (
        g["u_in"]
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["u_in_diff1_roll3_mean"] = (
        g["u_in_diff1"]
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["u_in_x_R"] = df["u_in"] * df["R"].astype(float)
    df["u_in_x_C"] = df["u_in"] * df["C"].astype(float)
    df["time_x_R"] = df["time_step"] * df["R"].astype(float)
    df["time_x_C"] = df["time_step"] * df["C"].astype(float)

    df["u_in_roll10_mean"] = (
        g["u_in"]
        .rolling(window=10, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )
    df["u_in_cum_mean"] = df["u_in_cum"] / (df["t_idx"].astype(float) + 1.0)

    return df


df_train_fe = add_features(df_train)
df_test_fe = add_features(df_test)

features_num = [
    "time_step",
    "u_in",
    "u_out",
    "u_in_lag1",
    "u_in_lag2",
    "u_in_diff1",
    "u_in_diff2",
    "u_in_cum",
    "u_in_x_time",
    "dt",
    "u_out_lag1",
    "u_out_cum",
    "u_in_area",
    "u_in_over_R",
    "u_in_over_C",
    "time_over_R",
    "time_over_C",
    "is_insp",
    "t_norm",
    "u_in_cum_insp",
    "t_idx",
    "u_in_cum_sq",
    "time_since_u_out_change",
    "insp_count",
    "u_in_roll3_mean",
    "u_in_diff1_roll3_mean",
    "u_in_x_R",
    "u_in_x_C",
    "time_x_R",
    "time_x_C",
    "u_in_roll10_mean",
    "u_in_cum_mean",
]
features_cat = ["R", "C"]

# CHANGE (score): Train on ALL rows to reduce train/test distribution mismatch,
# but upweight inspiratory rows to stay aligned with MAE being computed only there.
X_train = df_train_fe[features_num + features_cat]
y_train = df_train_fe["pressure"].astype(float)
sample_weight = np.where(df_train_fe["u_out"].values == 0, 5.0, 1.0).astype(float)

df_test_fe_id = df_test_fe.set_index("id", drop=False)
sub_id = sub.set_index("id", drop=False)
df_test_fe_aligned = df_test_fe_id.reindex(sub_id.index)
if df_test_fe_aligned.isnull().any().any():
    raise ValueError(
        "Alignment by id introduced NaNs; check id uniqueness/mismatch between test and sample_submission."
    )

X_test = df_test_fe_aligned[features_num + features_cat]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            features_num,
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
    ],
    remainder="drop",
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("reg", Ridge(alpha=1.0, random_state=0)),
    ]
)

model.fit(X_train, y_train, reg__sample_weight=sample_weight)

# CHANGE (score): Apply a simple global bias correction estimated on training inspiratory rows
# to better match the metric without altering model structure.
train_pred = model.predict(X_train).astype(float)
insp_mask = df_train_fe["u_out"].values == 0
bias = np.median((y_train.values[insp_mask] - train_pred[insp_mask]).astype(float))

pred = model.predict(X_test).astype(float) + float(bias)

pred_df = df_test_fe_aligned[["breath_id", "time_step", "u_out"]].copy()
pred_df["pred"] = pred
pred_df.sort_values(["breath_id", "time_step"], inplace=True)
pred_df["pred_insp_only"] = pred_df["pred"].where(pred_df["u_out"].values == 0)
pred_df["pred_filled"] = pred_df.groupby("breath_id")["pred_insp_only"].ffill()
pred_df["pred_filled"] = pred_df["pred_filled"].fillna(pred_df["pred"])
pred_df.sort_index(inplace=True)  # back to aligned-by-id row order

pred_final = pred_df["pred_filled"].to_numpy(dtype=float)
pred_final = np.array([find_nearest(p) for p in pred_final], dtype=float)

sub_out = sub_id.copy()
sub_out["pressure"] = pred_final
sub_out.reset_index(drop=True)[["id", "pressure"]].to_csv("submission.csv", index=False)

print(sub_out.reset_index(drop=True)[["id", "pressure"]].head())
print(
    "Wrote submission.csv with shape:",
    sub_out.reset_index(drop=True)[["id", "pressure"]].shape,
)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=1.43779e-19): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


   id   pressure
0   1   6.118700
1   2   7.595045
2   3   8.157463
3   4   9.071390
4   5  10.407131
Wrote submission.csv with shape: (603600, 2)
